# Solving Taxi-v3 with Monte Carlo Methods
### Reinforcement Learning Lab
---

#### Objectives:
1. Implement First-Visit and Every-Visit Monte Carlo Prediction.
2. Compare their performance on the Taxi-v3 environment.
3. Explore applications of Monte Carlo methods in RL.

#### Prerequisites:
- Basic Python programming.
- Understanding of RL concepts (states, actions, rewards, and policies).


In [ ]:
# Install required library
!pip install gym matplotlib -q

# Import libraries
import gym
import numpy as np
from collections import defaultdict
import matplotlib.pyplot as plt


In [ ]:
# Create the Taxi-v3 environment
env = gym.make("Taxi-v3")
env.reset()

# Render the environment
env.render()

print("Action Space:", env.action_space.n)
print("State Space:", env.observation_space.n)


In [ ]:
# Define a random policy
def random_policy(state):
    return env.action_space.sample()


In [ ]:
def mc_prediction_first_visit(env, policy, num_episodes, gamma=0.9):
    value_table = defaultdict(float)
    returns_sum = defaultdict(float)
    returns_count = defaultdict(float)

    for _ in range(num_episodes):
        state = env.reset()
        episode = []
        done = False
        
        while not done:
            action = policy(state)
            next_state, reward, done, _ = env.step(action)
            episode.append((state, action, reward))
            state = next_state
        
        G = 0
        visited_states = set()

        for t in reversed(range(len(episode))):
            state, _, reward = episode[t]
            G = gamma * G + reward
            if state not in visited_states:  # Only first visit
                visited_states.add(state)
                returns_sum[state] += G
                returns_count[state] += 1
                value_table[state] = returns_sum[state] / returns_count[state]
    
    return value_table


In [ ]:
def mc_prediction_every_visit(env, policy, num_episodes, gamma=0.9):
    value_table = defaultdict(float)
    returns_sum = defaultdict(float)
    returns_count = defaultdict(float)

    for _ in range(num_episodes):
        state = env.reset()
        episode = []
        done = False
        
        while not done:
            action = policy(state)
            next_state, reward, done, _ = env.step(action)
            episode.append((state, action, reward))
            state = next_state
        
        G = 0

        for t in reversed(range(len(episode))):
            state, _, reward = episode[t]
            G = gamma * G + reward
            returns_sum[state] += G
            returns_count[state] += 1
            value_table[state] = returns_sum[state] / returns_count[state]
    
    return value_table


In [ ]:
# Run First-Visit MC
num_episodes = 500
first_visit_values = mc_prediction_first_visit(env, random_policy, num_episodes)

# Run Every-Visit MC
every_visit_values = mc_prediction_every_visit(env, random_policy, num_episodes)

# Compare results for specific states
common_states = set(first_visit_values.keys()).intersection(every_visit_values.keys())
for state in list(common_states)[:5]:
    print(f"State {state}:")
    print(f"  First-Visit Value: {first_visit_values[state]:.2f}")
    print(f"  Every-Visit Value: {every_visit_values[state]:.2f}")


In [ ]:
# Visualize value estimates for a single state
def plot_value_estimates(first_visit, every_visit, state):
    plt.bar(["First-Visit", "Every-Visit"], 
            [first_visit.get(state, 0), every_visit.get(state, 0)])
    plt.title(f"Value Estimates for State {state}")
    plt.ylabel("Value Estimate")
    plt.show()

# Plot for a specific state
state_to_plot = list(common_states)[0]
plot_value_estimates(first_visit_values, every_visit_values, state_to_plot)


### Key Discussion Points:
1. **First-Visit MC**:
   - Updates state values based only on the first occurrence in an episode.
   - Avoids bias from over-sampling highly recurrent states.
   - May converge more slowly if states are rarely visited early.

2. **Every-Visit MC**:
   - Updates state values every time a state is visited in an episode.
   - Can result in faster convergence due to more updates but might over-sample recurrent states.

### Student Task:
- Analyze the differences in the outputs for specific states.
- Explain why one method might produce higher or lower value estimates than the other.
